# Performances of 2D integration vs 1D integration

This is dependent on:
* Number of azimuthal bins
* Pixel splitting
* Algorithm
* Implementation (i.e. programming language)
* Hardware used

Thus there is no general answer. But here is a quick benchmark to evaluate the penalty on performances:

import sys
import os
import time
import numpy
import fabio
import pyFAI
from pyFAI.test.utilstest import UtilsTest
import pyFAI.method_registry
import pyFAI.integrator.azimuthal
print(f"Python version: {sys.version}")
print(f"PyFAI version: {pyFAI.version}")
start_time = time.perf_counter()

In [1]:
import sys
import os
import time

os.environ["PYOPENCL_COMPILER_OUTPUT"] = "0"
start_time = time.perf_counter()

In [2]:
import fabio
import pyFAI
from pyFAI.test.utilstest import UtilsTest
import pyFAI.method_registry
import pyFAI.integrator.azimuthal
print(f"Python version: {sys.version}")
print(f"PyFAI version: {pyFAI.version}")


Python version: 3.14.0 | packaged by conda-forge | (main, Oct 22 2025, 23:24:08) [GCC 14.3.0]
PyFAI version: 2026.8.0-dev0


In [3]:
print("Number of way to performing integration:", len(pyFAI.method_registry.IntegrationMethod.list_available()))

Number of way to performing integration: 95


In [4]:
ai = pyFAI.load(UtilsTest.getimage("Pilatus1M.poni"))
img = fabio.open(UtilsTest.getimage("Pilatus1M.edf")).data
ai

Detector Pilatus 1M	 PixelSize= 172µm, 172µm	 BottomRight (3)
Wavelength= 1.000000 Å
SampleDetDist= 1.583231e+00 m	PONI= 3.341702e-02, 4.122778e-02 m	rot1=0.006487  rot2=0.007558  rot3=0.000000 rad
DirectBeamDist= 1583.310 mm	Center: x=179.981, y=263.859 pix	Tilt= 0.571° tiltPlanRotation= 130.640° λ= 1.000Å

In [5]:
%%time
#Tune those parameters to match your needs:
kw1 = {"data": img, "npt":1000}
kw2 = {"data": img, "npt_rad":1000}
#Actual benchmark:
res = {}
for k,v in pyFAI.method_registry.IntegrationMethod._registry.items():
    print(k)
    if k.dim == 1:
        res[k] = %timeit -o ai.integrate1d(method=v, **kw1)
    else:
        res[k] = %timeit -o ai.integrate2d(method=v, **kw2)

Method(dim=1, split='no', algo='histogram', impl='python', target=None)


30.5 ms ± 75.5 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
Method(dim=2, split='no', algo='histogram', impl='python', target=None)


140 ms ± 286 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
Method(dim=1, split='no', algo='histogram', impl='cython', target=None)


11.3 ms ± 16.5 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
Method(dim=2, split='no', algo='histogram', impl='cython', target=None)


16.6 ms ± 41.1 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
Method(dim=1, split='bbox', algo='histogram', impl='cython', target=None)


...

242 ms ± 917 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)
Method(dim=1, split='full', algo='lut', impl='opencl', target=(1, 0))


4.7 ms ± 31.8 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
Method(dim=2, split='full', algo='lut', impl='opencl', target=(1, 0))


213 ms ± 21.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
Method(dim=1, split='full', algo='lut', impl='opencl', target=(2, 0))


3.38 ms ± 28.3 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
Method(dim=2, split='full', algo='lut', impl='opencl', target=(2, 0))


287 ms ± 14.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
CPU times: user 2h 3min 59s, sys: 7min 18s, total: 2h 11min 18s
Wall time: 7min 50s


In [6]:
print("-"*80)
print(f"{'Split':5s} | {'Algo':9s} | {'Impl':6s}| {'1d (ms)':8s} | {'2d (ms)':8s} | {'ratio':6s} | Device")
print("-"*80)
for k in res:
    if k.dim == 1:
        k1 = k
        k2 = k._replace(dim=2)
        if k2 in res:
            print(f"{k1.split:5s} | {k1.algo:9s} | {k1.impl:6s}| {res[k1].best*1000:8.3f} | {res[k2].best*1000:8.3f} | {res[k2].best/res[k1].best:6.1f} | ",
                    end="")
        if k.target:
            print(pyFAI.method_registry.IntegrationMethod._registry.get(k).target_name)
        else:
            print()
print("-"*80)

--------------------------------------------------------------------------------
Split | Algo      | Impl  | 1d (ms)  | 2d (ms)  | ratio  | Device
--------------------------------------------------------------------------------
no    | histogram | python|   30.362 |  139.981 |    4.6 | 
no    | histogram | cython|   11.259 |   16.571 |    1.5 | 
bbox  | histogram | cython|   26.497 |   32.741 |    1.2 | 
full  | histogram | cython|  137.891 |  273.874 |    2.0 | 
no    | csr       | cython|   10.560 |   15.204 |    1.4 | 
bbox  | csr       | cython|   12.045 |   13.600 |    1.1 | 
no    | csr       | python|   10.052 |   14.089 |    1.4 | 
bbox  | csr       | python|   13.155 |   17.567 |    1.3 | 
no    | csc       | cython|    6.917 |    9.625 |    1.4 | 
bbox  | csc       | cython|    9.214 |   12.761 |    1.4 | 
no    | csc       | python|   11.140 |   14.475 |    1.3 | 
bbox  | csc       | python|   14.893 |   21.896 |    1.5 | 
bbox  | lut       | cython|   14.317 |   12.107 |   

In [7]:
print(f"Total runtime: {time.perf_counter()-start_time:.3f}s")

Total runtime: 471.791s
